# Classification analysis (titanic data)

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image

from sklearn import tree
from sklearn.metrics import RocCurveDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

# Show version of scikit-learn
import sklearn
sklearn.__version__

## Import titanic data

In [ ]:
# Read and select variables
df_titanic_orig = pd.read_csv("titanic.csv", sep=",", encoding="utf-8")

# Number of rows and columns
print(df_titanic_orig.shape)

# First records
df_titanic_orig.head(5)

## Variable description

- PassengerId passenger identification number
- Survival survival status (0 = No; 1 = Yes)
- Pclass passenger class (1 = 1st; 2 = 2nd; 3 = 3rd)
- Name name
- Sex sex
- Age age 
- SibSp number of siblings/spouses aboard
- Parch number of parents/children aboard
- Ticket ticket number
- Fare passenger fare (British pound)
- Cabin cabin
- Embarked port of embarkation (C = Cherbourg; Q = Queenstown; S = Southampton)

In [ ]:
Image("img.jpg", width='800')

## Count and remove missing values

In [ ]:
# Count missing values
print(df_titanic_orig.isna().sum())

# Remove missing values
df_titanic = df_titanic_orig.dropna(subset=['Survived', 'Sex', 'Age', 'Pclass', 'Fare'])

## Barchart survival status count by gender

In [ ]:
# Create a pivot table
table = df_titanic[['Sex', 'Survived']].pivot_table(index='Sex', 
                                        columns=['Survived'], 
                                        aggfunc=len)

# Plot a stacked bar chart
table.plot(kind='bar', 
           stacked=True, 
           ylabel='Counts', 
           xlabel='Gender',
           title='Survival Status Count by Gender', 
           rot=0,
           figsize=(6,4))

plt.show()

## Pivot table

In [ ]:
# Using pivot_table to reshape the data and calculate means 
pd.pivot_table(df_titanic[['Survived',
                           'Age',
                           'Sex',
                           'Fare',
                           'Pclass']],
               index=['Survived', 'Sex'],
               values=['Age', 'Fare', 'Pclass'],
               aggfunc=(np.mean, 'count')).round(0)

## Transform nominal variable to matrix with 0/1 values

In [ ]:
male = pd.get_dummies(df_titanic, drop_first=False, columns=['Sex'])
male[['Sex_female', 'Sex_male']].head()

## Create binary variable 'Sex_male' (wth 0=no, 1=yes)

In [ ]:
df_titanic['Sex_male'] = male['Sex_male']
df_titanic.head()

## Classification Tree
For details see: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

### Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X_train, X_test, y_train, y_test = train_test_split(df_titanic[['Age', 
                                                                'Sex_male',
                                                                'Pclass',
                                                                'Fare']], 
                                                                df_titanic['Survived'], 
                                                                test_size=0.20, 
                                                                random_state=42)

# Show X_train
print('X_train:')
print(X_train.head(), '\n')

# Show y_train
print('y_train:')
print(y_train.head())

### Fit the classification tree model and make predictions

In [ ]:
# Initialize the classification tree model 
clf = DecisionTreeClassifier(random_state=20, 
                             max_depth=3)

# Train the classification tree model 
clf = clf.fit(X_train, y_train)

# Make model predictions
y_pred = clf.predict(X_test)
y_pred

### Show confusion matrix and classification report

In [ ]:
# Confusion matrix
print('Confusion matrix')
print(confusion_matrix(y_test, y_pred), '\n')

# Classification report
print('Classification report')
print(classification_report(y_test, y_pred))

## Task 1b: Comparison with 50/50 train-test split

In [ ]:
# Create train and test samples with 50/50 split
X_train_50, X_test_50, y_train_50, y_test_50 = train_test_split(df_titanic[['Age', 
                                                                            'Sex_male',
                                                                            'Pclass',
                                                                            'Fare']], 
                                                                df_titanic['Survived'], 
                                                                test_size=0.50, 
                                                                random_state=42)

# Initialize and train the classification tree model with 50/50 split
clf_50 = DecisionTreeClassifier(random_state=20, max_depth=3)
clf_50 = clf_50.fit(X_train_50, y_train_50)

# Make model predictions
y_pred_50 = clf_50.predict(X_test_50)

# Show confusion matrix and classification report for 50/50 split
print('Confusion matrix (50/50 split)')
print(confusion_matrix(y_test_50, y_pred_50), '\n')

print('Classification report (50/50 split)')
print(classification_report(y_test_50, y_pred_50))

### Explanation (Task 1b):
When changing the train-test split from 80/20 to 50/50, we observe changes in accuracy and recall. With the 80/20 split, the model has more training data (80%) to learn patterns, which typically results in better performance on the test set. With the 50/50 split, the model has less training data (only 50%), which may lead to slightly different accuracy and recall values. However, the 50/50 split provides a larger test set for evaluation. The differences in performance metrics reflect the trade-off between training set size (for learning) and test set size (for evaluation). Generally, with less training data, the model may show lower accuracy and recall, though this depends on the dataset characteristics and model complexity.

### Print text representation of the classification tree

In [ ]:
# Text representation of the classification tree
text_rep = tree.export_text(clf, 
                            feature_names=list(X_train.columns))

# Print text_representation
print(text_rep)

## Visualize the classification tree

In [ ]:
# For the meaning of numbers in boxes see root node
fig = plt.figure(figsize=(12,5))
tree_plot = tree.plot_tree(clf, 
                   feature_names=list(X_train.columns),  
                   class_names=['not survived', 'survived'],
                   filled=True,
                   fontsize=10,
                   label='root')

## Task 1c: Classification tree with different max_depth

In [ ]:
# Initialize the classification tree model with different max_depth
clf_depth5 = DecisionTreeClassifier(random_state=20, max_depth=5)

# Train the classification tree model 
clf_depth5 = clf_depth5.fit(X_train, y_train)

# Make model predictions
y_pred_depth5 = clf_depth5.predict(X_test)

# Text representation of the classification tree with max_depth=5
text_rep_depth5 = tree.export_text(clf_depth5, feature_names=list(X_train.columns))
print('Text representation (max_depth=5):')
print(text_rep_depth5)

# Visualize the classification tree with max_depth=5
fig = plt.figure(figsize=(16,8))
tree_plot_depth5 = tree.plot_tree(clf_depth5, 
                                  feature_names=list(X_train.columns),  
                                  class_names=['not survived', 'survived'],
                                  filled=True,
                                  fontsize=7,
                                  label='root')

### Explanation (Task 1c):
With max_depth=5 (instead of max_depth=3), the classification tree becomes significantly more complex. The text representation shows more levels of decision rules (5 levels instead of 3), creating a deeper hierarchical structure. The graphical visualization displays a much larger and more intricate tree with more leaf nodes and decision paths. This increased complexity allows the model to capture more nuanced patterns in the data, potentially improving accuracy on the training set. However, deeper trees also risk overfitting, where the model learns noise in the training data rather than generalizable patterns. The tree now includes more combinations of features (Age, Sex_male, Pclass, Fare) to make finer-grained predictions about survival.

## Random Forest Classifier
For details see: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

### Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X2_train, X2_test, y2_train, y2_test = train_test_split(df_titanic[['Age', 
                                                                    'Sex_male',
                                                                    'Pclass',
                                                                    'Fare']], 
                                                        df_titanic['Survived'], 
                                                        test_size=0.20, 
                                                        random_state=42)

# Show X2_train
print('X2_train:')
print(X2_train.head(), '\n')

# Show y2_train
print('y2_train:')
print(y2_train.head())

### Fit the Random Forest Classifier

In [ ]:
# Initialize the random forest classifier
rfc = RandomForestClassifier(random_state=20, max_depth=10)

# Train the random forest classifier
rfc = rfc.fit(X2_train, y2_train)

# Predict the target variable (0 = not survived, 1 = survived)
y_pred_rf = rfc.predict(X2_test)

print('Predicted target variable (0 = not survived, 1 = survived)')
y_pred_rf

### Show confusion matrix and classification report

In [ ]:
# Confusion matrix
print('Confusion matrix')
print(confusion_matrix(y2_test, y_pred_rf), '\n')

# Classification report
print('Classification report')
print(classification_report(y2_test, y_pred_rf))

### Show feature importance

In [ ]:
cols = X2_train.columns

# Derive feature importance from random forest
importances = rfc.feature_importances_
std = np.std([tree.feature_importances_ for tree in rfc.estimators_], axis=0)
indices = np.argsort(importances)[::-1]

# Print col-names and importances-values
print( cols[indices] )
print( importances[indices] )

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances', 
           x='features', 
           color='darkred', 
           figsize=(6,3))

## Task 1d: Random Forest without Age and Sex_male variables

In [ ]:
# Create train and test samples WITHOUT Age and Sex_male
X3_train, X3_test, y3_train, y3_test = train_test_split(df_titanic[['Pclass', 'Fare']], 
                                                        df_titanic['Survived'], 
                                                        test_size=0.20, 
                                                        random_state=42)

# Initialize the random forest classifier
rfc_no_age_sex = RandomForestClassifier(random_state=20, max_depth=10)

# Train the random forest classifier
rfc_no_age_sex = rfc_no_age_sex.fit(X3_train, y3_train)

# Predict the target variable
y_pred_rf_no_age_sex = rfc_no_age_sex.predict(X3_test)

# Show confusion matrix and classification report
print('Confusion matrix (without Age and Sex_male)')
print(confusion_matrix(y3_test, y_pred_rf_no_age_sex), '\n')

print('Classification report (without Age and Sex_male)')
print(classification_report(y3_test, y_pred_rf_no_age_sex))

# Show feature importance
cols_no_age_sex = X3_train.columns
importances_no_age_sex = rfc_no_age_sex.feature_importances_
indices_no_age_sex = np.argsort(importances_no_age_sex)[::-1]

print('\nFeature importances (without Age and Sex_male):')
print(cols_no_age_sex[indices_no_age_sex])
print(importances_no_age_sex[indices_no_age_sex])

# Barplot with feature importance
df_fi_no_age_sex = pd.DataFrame({'features':cols_no_age_sex,'importances': importances_no_age_sex})
df_fi_no_age_sex.sort_values('importances', inplace=True)
df_fi_no_age_sex.plot(kind='barh', 
                      y='importances', 
                      x='features', 
                      color='darkred', 
                      figsize=(6,3))
plt.title('Feature Importance (without Age and Sex_male)')
plt.show()

### Explanation (Task 1d):
After removing the Age and Sex_male variables, the most important feature is Pclass (passenger class). This makes sense because, in the original model with all features, Sex_male was the most important predictor of survival, followed by Age. With these two critical features removed, Pclass becomes the dominant predictor. Pclass reflects the socioeconomic status of passengers and their location on the ship, which significantly affected survival rates during the Titanic disaster (first-class passengers had better access to lifeboats). Fare is the second most important feature, which is correlated with Pclass but provides additional information about the ticket price paid by passengers.

### ROC curve and AUC

In [ ]:
# Plot ROC curve and calculate AUC
plt.figure(figsize=(6,4))
ax = plt.gca()
rfc_disp = RocCurveDisplay.from_estimator(rfc, 
                                          X2_test, 
                                          y2_test, 
                                          ax=ax,
                                          alpha=0.8,
                                          c="darkred")
plt.show()

## Task 1e: ROC curve comparison - with and without Age and Sex_male

In [ ]:
# Plot ROC curves for both models side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve with all features (Age, Sex_male, Pclass, Fare)
rfc_disp_full = RocCurveDisplay.from_estimator(rfc, 
                                               X2_test, 
                                               y2_test, 
                                               ax=ax1,
                                               alpha=0.8,
                                               c="darkred")
ax1.set_title('ROC Curve - With Age and Sex_male')

# ROC curve without Age and Sex_male (only Pclass, Fare)
rfc_disp_no_age_sex = RocCurveDisplay.from_estimator(rfc_no_age_sex, 
                                                     X3_test, 
                                                     y3_test, 
                                                     ax=ax2,
                                                     alpha=0.8,
                                                     c="darkblue")
ax2.set_title('ROC Curve - Without Age and Sex_male')

plt.tight_layout()
plt.show()

# Print AUC values for comparison
from sklearn.metrics import roc_auc_score

# Calculate AUC for model with all features
y_pred_proba_full = rfc.predict_proba(X2_test)[:, 1]
auc_full = roc_auc_score(y2_test, y_pred_proba_full)

# Calculate AUC for model without Age and Sex_male
y_pred_proba_no_age_sex = rfc_no_age_sex.predict_proba(X3_test)[:, 1]
auc_no_age_sex = roc_auc_score(y3_test, y_pred_proba_no_age_sex)

print(f'AUC with Age and Sex_male: {auc_full:.4f}')
print(f'AUC without Age and Sex_male: {auc_no_age_sex:.4f}')
print(f'Difference in AUC: {auc_full - auc_no_age_sex:.4f}')

### Explanation (Task 1e):
The ROC curve and AUC values show a significant difference between the two models. The model WITH Age and Sex_male features has a higher AUC value (closer to 1.0), indicating better classification performance. The ROC curve for this model is positioned higher and further to the left, demonstrating superior true positive rate across different threshold values.

When Age and Sex_male are REMOVED, the AUC decreases notably. This is because:
1. Sex_male was the most important predictor in the original model - gender played a crucial role in survival ("women and children first" policy)
2. Age was also an important predictor - children had higher survival rates
3. Without these powerful predictors, the model relies only on Pclass and Fare, which are less discriminative

The reduced AUC indicates that the model without Age and Sex_male has lower overall discriminatory power, making it less accurate at distinguishing between survivors and non-survivors. This demonstrates the importance of including relevant demographic features in survival prediction models.

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')